In [14]:
# Import required libraries
import pandas as pd
import os

DATA_FOLDER = "/Users/shilppatel/Desktop/AA_FINAL_PROJECT"

# Load the airport reference dataset from the raw backup file
df_airport = pd.read_csv(os.path.join(DATA_FOLDER, "AIRPORT_INFO_backup.csv"), low_memory=False)

print(f"Shape: {df_airport.shape}")
print(f"Columns: {df_airport.columns.tolist()}")

Shape: (4565, 17)
Columns: ['AIRPRT_CD', 'AIRPRT_NM', 'CITY_METRO_IATA_CD', 'STATE_PROVNC_CD', 'CNTRY_CD', 'WRLD_AREA_DOT_CD', 'LAT_DEGREE_QTY', 'LAT_MINUTE_QTY', 'LAT_SECOND_QTY', 'LAT_HEMSPHR_CD', 'LNGTD_DEGREE_QTY', 'LNGTD_MINUTE_QTY', 'LNGTD_SECOND_QTY', 'LNGTD_HEMSPHR_CD', 'LNGST_RUNWAY_FT_QTY', 'ELEVATN_FT_QTY', 'SF_LOAD_TMS']


In [15]:
df_airport.head(50)

,AIRPRT_CD,AIRPRT_NM,CITY_METRO_IATA_CD,STATE_PROVNC_CD,CNTRY_CD,WRLD_AREA_DOT_CD,LAT_DEGREE_QTY,LAT_MINUTE_QTY,LAT_SECOND_QTY,LAT_HEMSPHR_CD,LNGTD_DEGREE_QTY,LNGTD_MINUTE_QTY,LNGTD_SECOND_QTY,LNGTD_HEMSPHR_CD,LNGST_RUNWAY_FT_QTY,ELEVATN_FT_QTY,SF_LOAD_TMS
0,OAH,Shindand Air Base,OAH,NaN,AF,701,33,23,32,N,62,15,40,E,7900,3780,2025-02-17 16:55:48.934
1,REX,Lucio Blanco Intl,REX,NaN,MX,148,26,0,30,N,98,13,41,W,6200,139,2025-02-17 16:55:48.934
2,YQX,Gander Intl.,YQX,NL,CA,961,48,56,13,N,54,34,5,W,10200,496,2025-02-17 16:55:48.934
3,MLW,Spriggs Payne,MLW,NaN,LR,537,6,17,19,N,10,45,32,W,6000,26,2025-02-17 16:55:48.934
4,CIJ,Capitan Anibal Arab,CIJ,NaN,BO,312,11,2,16,S,68,47,0,W,8500,805,2025-02-17 16:55:48.934
5,KGJ,Karonga Airport,KGJ,NaN,MW,542,9,57,12,S,33,53,35,E,5500,0,2025-02-17 16:55:48.934
6,FDF,Martinique A. Cesaire,FDF,NaN,MQ,252,14,35,32,N,60,59,47,W,9800,16,2025-02-17 16:55:48.934
7,IUE,Niue Island Intl.,IUE,NaN,NU,852,19,4,41,S,169,55,36,W,7600,211,2025-02-17 16:55:48.934
8,LTI,Altai Airport,LTI,NaN,MN,751,46,22,37,N,96,13,2,E,9500,7280,2025-02-17 16:55:48.934
9,LAP,Manuel Marquez de Leon,LAP,NaN,MX,148,24,4,21,N,110,21,45,W,8200,69,2025-02-17 16:55:48.934


In [16]:
# Drop columns that provide no value for modeling
df_airport.drop(columns=['STATE_PROVNC_CD', 'SF_LOAD_TMS'], inplace=True)

print(f"Columns remaining: {df_airport.shape[1]}")

Columns remaining: 15


In [17]:
# Convert latitude and longitude from degrees/minutes/seconds to decimal format
df_airport['airport_latitude'] = (
    df_airport['LAT_DEGREE_QTY'] +
    df_airport['LAT_MINUTE_QTY'] / 60 +
    df_airport['LAT_SECOND_QTY'] / 3600
)
df_airport.loc[df_airport['LAT_HEMSPHR_CD'] == 'S', 'airport_latitude'] *= -1

df_airport['airport_longitude'] = (
    df_airport['LNGTD_DEGREE_QTY'] +
    df_airport['LNGTD_MINUTE_QTY'] / 60 +
    df_airport['LNGTD_SECOND_QTY'] / 3600
)
df_airport.loc[df_airport['LNGTD_HEMSPHR_CD'] == 'W', 'airport_longitude'] *= -1

# Drop the raw coordinate columns now that decimal versions are created
df_airport.drop(columns=[
    'LAT_DEGREE_QTY', 'LAT_MINUTE_QTY', 'LAT_SECOND_QTY', 'LAT_HEMSPHR_CD',
    'LNGTD_DEGREE_QTY', 'LNGTD_MINUTE_QTY', 'LNGTD_SECOND_QTY', 'LNGTD_HEMSPHR_CD'
], inplace=True)

# Round to 5 decimal places
df_airport['airport_latitude'] = df_airport['airport_latitude'].round(5)
df_airport['airport_longitude'] = df_airport['airport_longitude'].round(5)

print(f"Columns remaining: {df_airport.shape[1]}")
print(df_airport[['AIRPRT_CD', 'airport_latitude', 'airport_longitude']].head(3))

Columns remaining: 9
  AIRPRT_CD  airport_latitude  airport_longitude
0       OAH          33.39222           62.26111
1       REX          26.00833          -98.22806
2       YQX          48.93694          -54.56806


In [18]:
# Check for airports with zero or suspiciously small runway lengths
# Zero runway length is clearly a data entry error - every airport has a physical runway
print(df_airport[['AIRPRT_NM', 'AIRPRT_CD', 'LNGST_RUNWAY_FT_QTY']]
      .sort_values('LNGST_RUNWAY_FT_QTY')
      .head(10)
      .to_string())

                 AIRPRT_NM AIRPRT_CD  LNGST_RUNWAY_FT_QTY
411         Bagram Airport       OAI                    0
720          Amata Airport       AMT                    0
2039                Matari       IRP                    0
1222    Gangneung Air Base       KAG                    0
1292  Boigu Island Airport       GIC                 2300
3757     Walter J. Koladza       GBR                 2500
1210    Natuashish Airport       YNP                 2500
3437    San Carlos Airport       SQL                 2600
1891       Paamiut Airport       JFR                 2600
382         Wairoa Airport       WIR                 2900


In [19]:
# The four airports above show zero runway length which is impossible
# Correct values were looked up manually from official aviation sources
runway_fixes = {
    'OAI': 12220,
    'AMT': 4000,
    'IRP': 8202,
    'KAG': 9000
}

for code, length in runway_fixes.items():
    df_airport.loc[df_airport['AIRPRT_CD'] == code, 'LNGST_RUNWAY_FT_QTY'] = length

print("Runway lengths corrected.")
print(df_airport[df_airport['AIRPRT_CD'].isin(runway_fixes.keys())][['AIRPRT_CD', 'AIRPRT_NM', 'LNGST_RUNWAY_FT_QTY']])

Runway lengths corrected.
     AIRPRT_CD           AIRPRT_NM  LNGST_RUNWAY_FT_QTY
411        OAI      Bagram Airport                12220
720        AMT       Amata Airport                 4000
1222       KAG  Gangneung Air Base                 9000
2039       IRP              Matari                 8202


In [20]:
# Save the cleaned airport dataset
output_path = os.path.join(DATA_FOLDER, "AIRPORT_INFO_clean.csv")
df_airport.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Shape: {df_airport.shape}")
print(f"Columns: {df_airport.columns.tolist()}")

Saved: /Users/shilppatel/Desktop/AA_FINAL_PROJECT/AIRPORT_INFO_clean.csv
Shape: (4565, 9)
Columns: ['AIRPRT_CD', 'AIRPRT_NM', 'CITY_METRO_IATA_CD', 'CNTRY_CD', 'WRLD_AREA_DOT_CD', 'LNGST_RUNWAY_FT_QTY', 'ELEVATN_FT_QTY', 'airport_latitude', 'airport_longitude']
